# 4.4 Demo - Deploying a DAB to Multiple Environments
**Lesson:** Variable Substitutions in DABs

One code base, several environments. Everything that differs is a **variable** or a **target override**:

| Mechanism | Example |
|---|---|
| Custom variables | `${var.schema}`: declared under `variables:`, with a default, overridden per target or with `--var` / `BUNDLE_VAR_<name>` |
| Built-in references | `${bundle.name}`, `${bundle.target}`, `${workspace.current_user.userName}`, `${workspace.file_path}` |
| Resource references | `${resources.pipelines.orders_pipeline.id}`, resolved at deploy time |
| Target overrides | Any section (resources, workspace, permissions, run_as) can be overridden inside a target |
| `mode` | `development`: name prefix, paused schedules, development pipelines. `production`: strict checks |

The project: `bundles/02_multi_env_dab`. A serverless **SDP pipeline** (Course 1 style) and a 3-task **job** (land data → refresh pipeline → report), deployed to `dev` and `stage`.

In [ ]:
from adelab import bundles
from adelab.spark import PROJECT_ROOT

BUNDLE = PROJECT_ROOT / "04-automated-deployment-dabs" / "bundles" / "02_multi_env_dab"
status = bundles.workspace_status()
bundles.tree(BUNDLE)
print((BUNDLE / "databricks.yml").read_text())

In [ ]:
print((BUNDLE / "resources" / "orders_job.yml").read_text())
print((BUNDLE / "resources" / "orders_pipeline.yml").read_text())

## A. The same bundle, resolved for each target
Compare names, schemas, paths, and the schedule's `pause_status`:

![One bundle resolved into dev, stage and prod with variables and target overrides](../images/4.4-targets-resolution.svg)

**How to read it:** the blue box is the YAML you just printed, with `${...}` placeholders. Values come from variable defaults, then the
which the next cell prints. Prod (dashed) follows the same pattern in `bundles/03_cicd_dab`.

In [ ]:
for target in ("dev", "stage"):
    cfg = bundles.preview(BUNDLE, target, show=False)
    job, pipe = cfg["resources"]["jobs"]["orders_job"], cfg["resources"]["pipelines"]["orders_pipeline"]
    print(f"── {target} (mode: {cfg['bundle'].get('mode')})")
    print("   job name     :", job["name"])
    print("   schedule     :", job["schedule"].get("pause_status", "UNPAUSED"))
    print("   pipeline     :", pipe["name"], "->", f"{pipe['catalog']}.{pipe['schema']}", "| development:", pipe.get("development", False))
    print("   landing path :", pipe["configuration"]["landing_path"])
    print("   rows per run :", job["tasks"][0]["notebook_task"]["base_parameters"]["rows"])

## B. Override a variable from the command line
`--var="name=value"` (or the environment variable `BUNDLE_VAR_orders_per_run=5`) wins over target values. CI systems use this to inject environment-specific values.

In [ ]:
cfg = bundles.preview(BUNDLE, "dev", variables={"orders_per_run": 5}, show=False)
print(cfg["resources"]["jobs"]["orders_job"]["tasks"][0]["notebook_task"]["base_parameters"])

In [ ]:
if bundles.require_workspace(status):
    bundles.cli("bundle", "validate", "-t", "dev", "--var", "orders_per_run=5", cwd=BUNDLE)

## C. Deploy to dev, run, then promote to stage
In real life `stage` is deployed by CI with a service principal (lesson 4.7). Deploying it yourself here shows the differences.

![Comparison of mode development and mode production](../images/4.4-dev-vs-prod-mode.svg)

**How to read it:** each row is one thing `mode` changes, independent of variables. `-t dev` gives you the left column (prefixed names,
paused schedule, your user folder); `-t stage` gives you the right one (plain names, active schedule, shared
`root_path`). That is why the stage job runs on its own schedule until you destroy it.

In [ ]:
if bundles.require_workspace(status):
    bundles.cli("bundle", "deploy", "-t", "dev", cwd=BUNDLE)
    bundles.cli("bundle", "run", "-t", "dev", "orders_job", cwd=BUNDLE)

In [ ]:
if bundles.require_workspace(status):
    bundles.cli("bundle", "deploy", "-t", "stage", cwd=BUNDLE)
    bundles.cli("bundle", "summary", "-t", "stage", cwd=BUNDLE)

In the workspace you now have `[dev <you>] orders_job_dev` (schedule paused) **and** `orders_job_stage` (schedule active!), each writing to its own schema. Clean up:

In [ ]:
if bundles.require_workspace(status):
    for t in ("stage", "dev"):
        bundles.cli("bundle", "destroy", "-t", t, "--auto-approve", cwd=BUNDLE)